# 04 – Default Model

**Goal:** Predict whether a loan will default, using only information known at origination.

**Setup:**
- 36-month mature loans (60-month loans issued in 2014-2015 had not matured)
- Time-based split: trained on loans issued 2007-2014, tested on loans issued in 2015
- Model A: borrower information only · Model B: borrower information + Lending Club grade
- Metric: AUC (ROC curve), because only about 15% of loans default

In [1]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from sqlalchemy import create_engine

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

load_dotenv("../.env")
engine = create_engine(os.getenv("DATABASE_URL"))

IMAGES_DIR = Path("../images")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

MATURE_COLOR = "#1f5fbf"
ALL_COLOR = "#9a9a9a"
GAP_COLOR = "#e8743b"


def save_fig(fig, name):
    fig.savefig(IMAGES_DIR / f"{name}.png", bbox_inches="tight", dpi=150)

## 2. Data
36-month mature loans with origination information only. Outcome columns are not loaded (data leakage).

In [2]:
DATA_QUERY = """
SELECT
    l.id,
    d.year,
    l.grade,
    l.sub_grade,
    l.int_rate,
    l.loan_amnt,
    l.annual_inc,
    l.dti,
    l.fico_avg,
    l.revol_util,
    l.inq_last_6mths,
    l.open_acc,
    l.emp_length_years,
    l.home_ownership,
    l.verification_status,
    l.purpose,
    l.is_default
FROM loans_mature AS l
JOIN dim_date AS d ON l.issue_date = d.month_date
WHERE l.term_months = 36
"""

loans = pd.read_sql(DATA_QUERY, engine)
print(loans.shape)
print(f"Default rate: {loans['is_default'].mean():.2%}")

(621016, 17)
Default rate: 13.95%


## 3. Features
- `emp_length_missing`: 1 if employment length was not reported (20.86% vs 14.51% default in EDA)
- `loan_to_income`: loan amount / annual income (the same loan is heavier for a lower income)
- `log_annual_inc`: income is very skewed (up to $11M), so its logarithm is used

In [3]:
loans["emp_length_missing"] = loans["emp_length_years"].isna().astype(int)
loans["loan_to_income"] = loans["loan_amnt"] / loans["annual_inc"]
loans["log_annual_inc"] = np.log(loans["annual_inc"])

loans[["emp_length_missing", "loan_to_income", "log_annual_inc"]].describe().round(3)

,emp_length_missing,loan_to_income,log_annual_inc
count,621016.000,621016.000,621016.000
mean,0.058,0.196,11.029
std,0.234,0.106,0.549
min,0.000,0.000,7.090
25%,0.000,0.114,10.669
50%,0.000,0.180,11.002
75%,0.000,0.266,11.374
max,1.000,10.000,16.013


## 4. Time-based split
The model is trained on older loans and tested on newer loans, as it would be used in practice.
A random split would mix years and give an over-optimistic result.

In [4]:
train = loans[loans["year"] <= 2014]
test = loans[loans["year"] == 2015]

summary = pd.DataFrame({
    "loans": [len(train), len(test)],
    "default_rate_pct": [train["is_default"].mean() * 100, test["is_default"].mean() * 100],
}, index=["train (2007-2014)", "test (2015)"]).round(2)
summary

,loans,default_rate_pct
train (2007-2014),337992,13.16
test (2015),283024,14.89
